In [1]:
import os
import re
import glob

import numpy as np
import pandas as pd

import matplotlib

# Utilizar un backend que no necesita abrir ventanas
matplotlib.use("Agg")

import matplotlib.pyplot as plt

from mpl_toolkits.mplot3d import Axes3D

from sklearn.cluster import DBSCAN

from scipy.optimize import linear_sum_assignment

import cv2


# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

INPUT_DIR = "pointclouds"

OUTPUT_DIR = "output"

VIDEO_PATH = os.path.join(
    OUTPUT_DIR,
    "tracking_3d.mp4"
)

# Resolución del vídeo
VIDEO_WIDTH = 1280
VIDEO_HEIGHT = 720

# FPS
FPS = 10


# ============================================================
# CONFIGURACIÓN DEL CLUSTERING
# ============================================================

# Distancia máxima entre puntos vecinos
DBSCAN_EPS = 1

# Puntos mínimos para considerar una región densa
DBSCAN_MIN_SAMPLES = 8

# Número mínimo de puntos que debe tener un cluster
MIN_CLUSTER_POINTS = 50

# Número máximo de puntos de un cluster
MAX_CLUSTER_POINTS = 100000


# ============================================================
# CONFIGURACIÓN DEL TRACKING
# ============================================================

# Distancia máxima permitida para asociar un cluster
# con un vehículo existente
MAX_TRACKING_DISTANCE = 4.0

# Frames que un vehículo puede desaparecer temporalmente
MAX_MISSED_FRAMES = 5


# ============================================================
# FILTRO ESPACIAL
# ============================================================

# Estos límites debes adaptarlos a tus datos.
# Son los límites de la zona que se mostrará.

X_MIN = -50
X_MAX = 50

Y_MIN = -20
Y_MAX = 20

Z_MIN = -10
Z_MAX = 10


# ============================================================
# CÁMARA 3D
# ============================================================

# Elevación de la cámara.
# 90 sería totalmente desde arriba.
# 30-45 produce una vista diagonal.

CAMERA_ELEVATION = 35

# Ángulo horizontal de la cámara
CAMERA_AZIMUTH = -60


# ============================================================
# CLASE TRACK
# ============================================================

class Track:

    def __init__(self, track_id, centroid, points):

        self.id = track_id

        self.centroid = np.array(
            centroid,
            dtype=float
        )

        self.previous_centroid = np.array(
            centroid,
            dtype=float
        )

        self.velocity = np.zeros(3)

        self.points = points

        self.missed_frames = 0

        self.age = 1

    def update(self, centroid, points):

        centroid = np.array(
            centroid,
            dtype=float
        )

        self.previous_centroid = self.centroid.copy()

        self.velocity = centroid - self.centroid

        self.centroid = centroid

        self.points = points

        self.missed_frames = 0

        self.age += 1

    def mark_missed(self):

        self.missed_frames += 1


# ============================================================
# TRACKER
# ============================================================

class ObjectTracker:

    def __init__(self):

        self.tracks = {}

        self.next_id = 1

    def create_track(self, detection):

        track = Track(
            self.next_id,
            detection["centroid"],
            detection["points"]
        )

        self.tracks[self.next_id] = track

        self.next_id += 1

    def remove_old_tracks(self):

        ids_to_remove = []

        for track_id, track in self.tracks.items():

            if track.missed_frames > MAX_MISSED_FRAMES:

                ids_to_remove.append(track_id)

        for track_id in ids_to_remove:

            del self.tracks[track_id]

    def update(self, detections):

        """
        detections es una lista de diccionarios:

        {
            "centroid": np.array([x, y, z]),
            "points": np.array(...)
        }
        """

        # ----------------------------------------------------
        # PRIMER FRAME
        # ----------------------------------------------------

        if len(self.tracks) == 0:

            for detection in detections:

                self.create_track(detection)

            return self.tracks

        # ----------------------------------------------------
        # SI NO HAY DETECCIONES
        # ----------------------------------------------------

        if len(detections) == 0:

            for track in self.tracks.values():

                track.mark_missed()

            self.remove_old_tracks()

            return self.tracks

        track_ids = list(
            self.tracks.keys()
        )

        track_centroids = np.array(
            [
                self.tracks[track_id].centroid
                for track_id in track_ids
            ]
        )

        detection_centroids = np.array(
            [
                detection["centroid"]
                for detection in detections
            ]
        )

        # ----------------------------------------------------
        # MATRIZ DE DISTANCIAS
        # ----------------------------------------------------

        distance_matrix = np.zeros(
            (
                len(track_centroids),
                len(detection_centroids)
            )
        )

        for i, track_centroid in enumerate(
            track_centroids
        ):

            for j, detection_centroid in enumerate(
                detection_centroids
            ):

                distance_matrix[i, j] = np.linalg.norm(
                    track_centroid - detection_centroid
                )

        # ----------------------------------------------------
        # ASIGNACIÓN HÚNGARA
        # ----------------------------------------------------

        rows, cols = linear_sum_assignment(
            distance_matrix
        )

        matched_tracks = set()

        matched_detections = set()

        for row, col in zip(rows, cols):

            distance = distance_matrix[row, col]

            if distance <= MAX_TRACKING_DISTANCE:

                track_id = track_ids[row]

                self.tracks[track_id].update(
                    detection_centroids[col],
                    detections[col]["points"]
                )

                matched_tracks.add(row)

                matched_detections.add(col)

        # ----------------------------------------------------
        # TRACKS NO ASIGNADOS
        # ----------------------------------------------------

        for i, track_id in enumerate(track_ids):

            if i not in matched_tracks:

                self.tracks[track_id].mark_missed()

        # ----------------------------------------------------
        # NUEVAS DETECCIONES
        # ----------------------------------------------------

        for i, detection in enumerate(detections):

            if i not in matched_detections:

                self.create_track(detection)

        # ----------------------------------------------------
        # ELIMINAR TRACKS ANTIGUOS
        # ----------------------------------------------------

        self.remove_old_tracks()

        return self.tracks


# ============================================================
# BUSCAR COLUMNAS X Y Z
# ============================================================

def find_xyz_columns(df):

    columns = {
        str(column).lower().strip(): column
        for column in df.columns
    }

    possible_x = [
        "x",
        "pos_x",
        "position_x",
        "coord_x",
        "point_x"
    ]

    possible_y = [
        "y",
        "pos_y",
        "position_y",
        "coord_y",
        "point_y"
    ]

    possible_z = [
        "z",
        "pos_z",
        "position_z",
        "coord_z",
        "point_z"
    ]

    x_column = None
    y_column = None
    z_column = None

    for name in possible_x:

        if name in columns:

            x_column = columns[name]

            break

    for name in possible_y:

        if name in columns:

            y_column = columns[name]

            break

    for name in possible_z:

        if name in columns:

            z_column = columns[name]

            break

    if x_column is None:

        raise ValueError(
            "No se ha encontrado la columna X"
        )

    if y_column is None:

        raise ValueError(
            "No se ha encontrado la columna Y"
        )

    if z_column is None:

        raise ValueError(
            "No se ha encontrado la columna Z"
        )

    return x_column, y_column, z_column


# ============================================================
# LEER CSV
# ============================================================

def load_pointcloud(csv_path):

    print(
        f"Leyendo: {os.path.basename(csv_path)}"
    )

    try:

        df = pd.read_csv(csv_path)

    except Exception:

        df = pd.read_csv(
            csv_path,
            sep=";"
        )

    x_col, y_col, z_col = find_xyz_columns(df)

    points = df[
        [
            x_col,
            y_col,
            z_col
        ]
    ].values.astype(
        np.float32
    )

    # Eliminar NaN e infinitos
    points = points[
        np.all(
            np.isfinite(points),
            axis=1
        )
    ]

    return points


# ============================================================
# FILTRAR NUBE
# ============================================================

def filter_pointcloud(points):

    if len(points) == 0:

        return points

    mask = np.ones(
        len(points),
        dtype=bool
    )

    mask &= points[:, 0] >= X_MIN
    mask &= points[:, 0] <= X_MAX

    mask &= points[:, 1] >= Y_MIN
    mask &= points[:, 1] <= Y_MAX

    mask &= points[:, 2] >= Z_MIN
    mask &= points[:, 2] <= Z_MAX

    return points[mask]


# ============================================================
# CLUSTERING DBSCAN
# ============================================================

def cluster_pointcloud(points):

    if len(points) < DBSCAN_MIN_SAMPLES:

        return []

    print(
        f"Ejecutando DBSCAN sobre "
        f"{len(points)} puntos..."
    )

    dbscan = DBSCAN(
        eps=DBSCAN_EPS,
        min_samples=DBSCAN_MIN_SAMPLES
    )

    labels = dbscan.fit_predict(
        points
    )

    detections = []

    unique_labels = np.unique(
        labels
    )

    for label in unique_labels:

        # -1 significa ruido
        if label == -1:

            continue

        cluster_points = points[
            labels == label
        ]

        number_points = len(
            cluster_points
        )

        # Eliminar clusters pequeños
        if number_points < MIN_CLUSTER_POINTS:

            continue

        # Eliminar clusters excesivamente grandes
        if number_points > MAX_CLUSTER_POINTS:

            continue

        centroid = np.mean(
            cluster_points,
            axis=0
        )

        detections.append(
            {
                "centroid": centroid,
                "points": cluster_points
            }
        )

    print(
        f"Clusters válidos: {len(detections)}"
    )

    return detections


# ============================================================
# COLOR DETERMINISTA PARA CADA ID
# ============================================================

def get_color(track_id):

    rng = np.random.default_rng(
        track_id * 12345
    )

    color = rng.integers(
        60,
        255,
        size=3
    )

    return (
        color[0] / 255.0,
        color[1] / 255.0,
        color[2] / 255.0
    )


# ============================================================
# DIBUJAR TRACKS EN 3D
# ============================================================

def draw_tracks_3d(ax, tracks):

    visible_tracks = 0

    for track_id, track in tracks.items():

        # No mostrar objetos perdidos
        if track.missed_frames > 0:

            continue

        points = track.points

        if points is None or len(points) == 0:

            continue

        visible_tracks += 1

        color = get_color(
            track_id
        )

        # ----------------------------------------------------
        # PUNTOS DEL VEHÍCULO
        # ----------------------------------------------------

        ax.scatter(
            points[:, 0],
            points[:, 1],
            points[:, 2],
            s=3,
            c=[color],
            alpha=0.8,
            depthshade=True
        )

        # ----------------------------------------------------
        # CENTROIDE
        # ----------------------------------------------------

        centroid = track.centroid

        ax.scatter(
            centroid[0],
            centroid[1],
            centroid[2],
            s=60,
            c="white",
            edgecolors=[color],
            linewidths=2,
            depthshade=True
        )

        # ----------------------------------------------------
        # ETIQUETA ID
        # ----------------------------------------------------

        ax.text(
            centroid[0],
            centroid[1],
            centroid[2] + 0.8,
            f"ID {track_id}",
            color="black",
            fontsize=11,
            fontweight="bold",
            bbox=dict(
                facecolor="white",
                alpha=0.8,
                edgecolor=color
            )
        )

    return visible_tracks


# ============================================================
# DIBUJAR INFORMACIÓN
# ============================================================

def draw_information(
    ax,
    frame_number,
    visible_tracks
):

    ax.text2D(
        0.02,
        0.95,
        f"Frame: {frame_number}    "
        f"Vehículos: {visible_tracks}",
        transform=ax.transAxes,
        fontsize=12,
        bbox=dict(
            facecolor="white",
            alpha=0.8
        )
    )


# ============================================================
# CREAR FRAME 3D
# ============================================================

def create_3d_frame(
    tracks,
    frame_number
):

    fig = plt.figure(
        figsize=(
            VIDEO_WIDTH / 100,
            VIDEO_HEIGHT / 100
        ),
        dpi=100
    )

    ax = fig.add_subplot(
        111,
        projection="3d"
    )

    # --------------------------------------------------------
    # DIBUJAR VEHÍCULOS
    # --------------------------------------------------------

    visible_tracks = draw_tracks_3d(
        ax,
        tracks
    )

    # --------------------------------------------------------
    # LÍMITES DE LOS EJES
    # --------------------------------------------------------

    ax.set_xlim(
        X_MIN,
        X_MAX
    )

    ax.set_ylim(
        Y_MIN,
        Y_MAX
    )

    ax.set_zlim(
        Z_MIN,
        Z_MAX
    )

    # --------------------------------------------------------
    # ETIQUETAS
    # --------------------------------------------------------

    ax.set_xlabel(
        "X"
    )

    ax.set_ylabel(
        "Y"
    )

    ax.set_zlabel(
        "Z"
    )

    ax.set_title(
        "Tracking 3D de vehículos"
    )

    # --------------------------------------------------------
    # CÁMARA DIAGONAL DESDE ARRIBA
    # --------------------------------------------------------

    ax.view_init(
        elev=CAMERA_ELEVATION,
        azim=CAMERA_AZIMUTH
    )

    # --------------------------------------------------------
    # ASPECTO DE LOS EJES
    # --------------------------------------------------------

    try:

        ax.set_box_aspect(
            (
                X_MAX - X_MIN,
                Y_MAX - Y_MIN,
                Z_MAX - Z_MIN
            )
        )

    except Exception:

        pass

    # --------------------------------------------------------
    # GRID
    # --------------------------------------------------------

    ax.grid(False)
    ax.set_axis_off()

    # --------------------------------------------------------
    # INFORMACIÓN
    # --------------------------------------------------------

    draw_information(
        ax,
        frame_number,
        visible_tracks
    )

    # --------------------------------------------------------
    # GUARDAR IMAGEN TEMPORAL
    # --------------------------------------------------------

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    image = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8
    )

    image = image.reshape(
        height,
        width,
        4
    )

    # Matplotlib usa RGBA.
    # OpenCV necesita BGR.
    image = cv2.cvtColor(
        image,
        cv2.COLOR_RGBA2BGR
    )

    # Asegurar tamaño exacto
    image = cv2.resize(
        image,
        (
            VIDEO_WIDTH,
            VIDEO_HEIGHT
        )
    )

    plt.close(
        fig
    )

    return image


# ============================================================
# ORDEN NATURAL DE LOS CSV
# ============================================================

def natural_sort_key(path):

    filename = os.path.basename(
        path
    )

    return [
        int(text)
        if text.isdigit()
        else text.lower()
        for text in re.split(
            r"(\d+)",
            filename
        )
    ]


# ============================================================
# PROCESAR FRAME
# ============================================================

def process_frame(
    csv_path,
    tracker,
    frame_number
):

    points = load_pointcloud(
        csv_path
    )

    print(
        f"Puntos originales: {len(points)}"
    )

    points = filter_pointcloud(
        points
    )

    print(
        f"Puntos tras el filtro: {len(points)}"
    )

    detections = cluster_pointcloud(
        points
    )

    tracks = tracker.update(
        detections
    )

    image = create_3d_frame(
        tracks,
        frame_number
    )

    return image


# ============================================================
# MAIN
# ============================================================

def main():

    print("=" * 60)
    print("CLUSTERING + TRACKING 3D")
    print("=" * 60)

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )

    # --------------------------------------------------------
    # BUSCAR CSV
    # --------------------------------------------------------

    csv_files = glob.glob(
        os.path.join(
            INPUT_DIR,
            "*.csv"
        )
    )

    if len(csv_files) == 0:

        print(
            f"No se encontraron CSV en "
            f"'{INPUT_DIR}'"
        )

        return

    csv_files.sort(
        key=natural_sort_key
    )

    print(
        f"CSV encontrados: {len(csv_files)}"
    )

    # --------------------------------------------------------
    # CREAR VIDEO
    # --------------------------------------------------------

    fourcc = cv2.VideoWriter_fourcc(
        *"mp4v"
    )

    video_writer = cv2.VideoWriter(
        VIDEO_PATH,
        fourcc,
        FPS,
        (
            VIDEO_WIDTH,
            VIDEO_HEIGHT
        )
    )

    if not video_writer.isOpened():

        print(
            "No se pudo crear el vídeo."
        )

        return

    # --------------------------------------------------------
    # TRACKER
    # --------------------------------------------------------

    tracker = ObjectTracker()

    # --------------------------------------------------------
    # PROCESAR TODOS LOS CSV
    # --------------------------------------------------------

    for frame_number, csv_path in enumerate(
        csv_files,
        start=1
    ):

        print("\n")
        print("=" * 60)
        print(
            f"FRAME {frame_number}/"
            f"{len(csv_files)}"
        )
        print("=" * 60)

        try:

            image = process_frame(
                csv_path,
                tracker,
                frame_number
            )

            video_writer.write(
                image
            )

            # Guardar imagen individual
            frame_path = os.path.join(
                OUTPUT_DIR,
                f"frame_{frame_number:06d}.png"
            )

            cv2.imwrite(
                frame_path,
                image
            )

        except Exception as error:

            print(
                f"Error procesando "
                f"{csv_path}: {error}"
            )

            continue

    # --------------------------------------------------------
    # CERRAR VIDEO
    # --------------------------------------------------------

    video_writer.release()

    print("\n")
    print("=" * 60)
    print("PROCESO TERMINADO")
    print("=" * 60)

    print(
        "Vídeo generado:"
    )

    print(
        os.path.abspath(
            VIDEO_PATH
        )
    )


# ============================================================
# EJECUTAR
# ============================================================

if __name__ == "__main__":

    main()

CLUSTERING + TRACKING 3D
CSV encontrados: 369


FRAME 1/369
Leyendo: pointcloud_1727346186_488981170.csv
Puntos originales: 12857
Puntos tras el filtro: 12391
Ejecutando DBSCAN sobre 12391 puntos...
Clusters válidos: 2


FRAME 2/369
Leyendo: pointcloud_1727346186_538003836.csv
Puntos originales: 12983
Puntos tras el filtro: 12527
Ejecutando DBSCAN sobre 12527 puntos...
Clusters válidos: 2


FRAME 3/369
Leyendo: pointcloud_1727346186_586942416.csv
Puntos originales: 13118
Puntos tras el filtro: 12647
Ejecutando DBSCAN sobre 12647 puntos...
Clusters válidos: 2


FRAME 4/369
Leyendo: pointcloud_1727346186_638212608.csv
Puntos originales: 13216
Puntos tras el filtro: 12738
Ejecutando DBSCAN sobre 12738 puntos...
Clusters válidos: 2


FRAME 5/369
Leyendo: pointcloud_1727346186_688448845.csv
Puntos originales: 13365
Puntos tras el filtro: 12896
Ejecutando DBSCAN sobre 12896 puntos...
Clusters válidos: 2


FRAME 6/369
Leyendo: pointcloud_1727346186_738318465.csv
Puntos originales: 13403
Punto